In [1]:

import os, sys, platform, subprocess, json, hashlib, importlib.metadata as md
import threadpoolctl
os.chdir(subprocess.run(['git','rev-parse','--show-toplevel'],capture_output=True,text=True).stdout.strip()); sys.path.insert(0, os.getcwd())
import sybil_pipeline as sp
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
cpu = open('/proc/cpuinfo').read().split('\n\n')[0]
fields = dict(l.split(':', 1) for l in cpu.splitlines() if ':' in l)
fields = {k.strip(): v.strip() for k, v in fields.items()}
env = dict(
  recorded_utc=sh('date -u +%FT%TZ'),
  source_commit_full=sh('git rev-parse HEAD'), code_commit=sp.CODE_COMMIT,
  python=platform.python_version(), python_full=sys.version, executable=sys.executable,
  machine=platform.machine(), processor=platform.processor(), system=platform.system(), kernel=platform.release(),
  uname=sh('uname -a'),
  cpu_model_name=fields.get('model name'), cpu_vendor=fields.get('vendor_id'), cpu_family=fields.get('cpu family'),
  cpu_model=fields.get('model'), cpu_stepping=fields.get('stepping'), cpu_microcode=fields.get('microcode'),
  cpu_mhz=fields.get('cpu MHz'), cpu_cache_size=fields.get('cache size'),
  cpu_flags_sha256=hashlib.sha256(' '.join(sorted(fields.get('flags','').split())).encode()).hexdigest(),
  cpu_flags=fields.get('flags'),
  os_cpu_count=os.cpu_count(), sched_affinity=len(os.sched_getaffinity(0)),
  cgroup_cpu_max=sh('cat /sys/fs/cgroup/cpu.max 2>/dev/null') or 'none',
  mem_total=sh("grep MemTotal /proc/meminfo"), cgroup_mem_max=sh('cat /sys/fs/cgroup/memory.max 2>/dev/null') or 'none',
  lscpu=sh('lscpu'),
  model_threads=dict(N_JOBS=sp.N_JOBS, LGBM_REPRO=sp.LGBM_REPRO, SEED=sp.SEED),
  thread_env={k: os.environ.get(k) for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS','PYTHONHASHSEED']},
  threadpools=threadpoolctl.threadpool_info(),
  packages=sorted(f"{d.metadata['Name']}=={d.version}" for d in md.distributions()),
)
for k in ['xgboost','lightgbm','scikit-learn','numpy','pandas','scipy','matplotlib','pyarrow','shap','nbconvert','nbclient','ipykernel']:
    print(f'{k:14s}', md.version(k))
print({k: env[k] for k in ['python','executable','machine','system','kernel','cpu_model_name','cpu_family','cpu_model','cpu_stepping','cpu_microcode','os_cpu_count','sched_affinity','cgroup_cpu_max','mem_total','code_commit','model_threads','thread_env']})
json.dump(env, open('logs/rerun_x86_68e8d63/run/env_after_restart.json', 'w'), indent=1)


xgboost        3.2.0
lightgbm       4.6.0
scikit-learn   1.8.0
numpy          2.4.6
pandas         3.0.6
scipy          1.17.1
matplotlib     3.11.2
pyarrow        25.0.1
shap           0.51.0
nbconvert      7.17.1
nbclient       0.11.0
ipykernel      7.4.0
{'python': '3.11.15', 'executable': '/home/user/paven86/venv311/bin/python', 'machine': 'x86_64', 'system': 'Linux', 'kernel': '6.18.44-fc-v80', 'cpu_model_name': 'Intel(R) Xeon(R) Processor @ 2.80GHz', 'cpu_family': '6', 'cpu_model': '85', 'cpu_stepping': '7', 'cpu_microcode': '0x1', 'os_cpu_count': 4, 'sched_affinity': 4, 'cgroup_cpu_max': 'none', 'mem_total': 'MemTotal:       16480952 kB', 'code_commit': '412db11', 'model_threads': {'N_JOBS': 4, 'LGBM_REPRO': {'force_col_wise': True, 'deterministic': True}, 'SEED': 42}, 'thread_env': {'OMP_NUM_THREADS': None, 'OPENBLAS_NUM_THREADS': None, 'MKL_NUM_THREADS': None, 'NUMEXPR_NUM_THREADS': None, 'PYTHONHASHSEED': None}}
